In [1]:
import pandas as pd
import json
from langdetect import detect
from langdetect import DetectorFactory
import re
import ast
DetectorFactory.seed = 0

In [40]:
sample_df = pd.read_csv("../data/raw/advanced_rag_full_dataset.csv")
sample_df.shape

(5021, 25)

In [37]:
def decode_unicode_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    if not text:
        return ""
    # Ignore malformed escape sequences (e.g., truncated \\uXXXX)
    decoded = text.encode("utf-8").decode("unicode_escape", errors="ignore")
    return decoded.encode('ascii', 'ignore').decode('ascii')

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    if not text:
        return ""
    text = decode_unicode_escape(text)
    # Remove non-UTF-8 characters
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    # normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    
    return text.strip()


def detect_language(text):
    if pd.isna(text):
        return "unknown"
    text = str(text).strip()
    if text == "":
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"
    
def parse_product_id(product_id):
    product_id = str(product_id).strip()
    product_id = product_id.encode("utf-8").decode("unicode_escape", errors="ignore")
    if product_id:
        try:
            return json.loads(product_id)
        except:
            try:
                return ast.literal_eval(product_id)
            except:
                return []
        

def process_row(row):
    return {
        "doc_id": row["id"],
        "title": row["title"],
        "file_name": row["file_name"],
        "product_id": parse_product_id(row["product_id"]),
        "file_content": clean_text(row["file_content"]),
        "document_type": row["document_type"],
        "language": detect_language(row["file_content"])
    }

In [38]:
docs = sample_df.apply(process_row, axis=1).tolist()
print(docs[0])
# with open("processed_docs.json", "w") as f:
#     json.dump(docs, f, indent=4)

{'doc_id': 'dm_0-7f4b0076d147778819ece847ef80f810', 'title': 'Universal CLSS Gateway FAQ', 'file_name': 'ba-fire-FAQ-Honeywell-Universal-CLSS-Gateway-revA.pdf', 'product_id': ['P1922910', 'P1921461'], 'file_content': '2023 Honeywell Inc. All rights reserved. | www.notifier.com.au \nPage 1 of 2 \nHoneywell Universal CLSS Gateway \nModbus and BACnet Feature \n \n \nFAQs \n \n1. Which panels will the CLSS Gateway Modbus and BACnet support? \nUniversal CLSS Gateway will support ONYX AFP-3030 and AFP-2800 Panels. \n \n \n2. Will there be a price increase for the CLSS Gateway Modbus and BACnet \nlicenses compared to legacy gateways? \nNo. there wont be any price increases however, youll be activating the relevant \nlicencing feature licence from the CLSS portal (dashboard or the App). \n \n \n3. Do the new licence features have a subscription fee? \nNo, these are onetime licence activations \n \n \n4. What regulatory certification will the CLSS Gateway - Modbus and BACnet \nhave upon release

In [41]:
docs[7]

{'doc_id': 'dm_50b8acd692b03421d858bea61f180a8f',
 'title': 'IQ5 DataSheet-IT',
 'file_name': 'HBA-BMS-IQ5-IT-DataSheet.pdf',
 'product_id': ['BRP900~IQ5-00-24'],
 'file_content': 'Scheda tecnica del controllore IQ5 TA201480ITA Versione 1, 16-ottobre-2023. Relativa a v1.00 1\nScheda tecnica\nIQ5\nControllore\nControllore IQ5\nDescrizione\nIl controllore IQ5 mette a disposizione una piattaforma di controllo \nsicura e versatile per i sistemi di gestione di edifici ed energia. \nCon tre porte Ethernet e tre porte RS-485 integrate, IQ5 non consente \nsoltanto di creare una potente rete Trend, ma offre la possibilit \ndi interfacciarsi con unampia gamma di dispositivi di terze parti \nutilizzando BACnet, Modbus, M-Bus, MSTP e molto altro.\nIQ5 si integra con i pi recenti moduli di ingresso/uscita IQ5-IO su un bus \nT1L ad alta velocit. disponibile anche un bus separato da utilizzare \ncon i moduli IO delle gamme IQ4 e XCITE. Le pratiche opzioni di licenza \nconsentono di configurare facilm